In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset

In [2]:
from fastai.text.all import *
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

In [5]:
from pathlib import Path
import pandas as pd

# Define the path
poem_path = Path("../../Dataset/poems.txt")

# Check if the file exists
if not poem_path.exists():
    raise FileNotFoundError(f"File not found at: {poem_path.resolve()}")

# Read the text content
poem_text = poem_path.read_text(encoding="utf-8")  # safer encoding

# Wrap into a pandas DataFrame
df = pd.DataFrame({'text': [poem_text]})


In [8]:
df.shape

(1, 1)

In [15]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token

def tokenize_function(example):
    return tokenizer(example["text"])


# Convert pandas DataFrame to Hugging Face Dataset
from datasets import Dataset
dataset = Dataset.from_pandas(df)

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (19896 > 1024). Running this sequence through the model will result in indexing errors


In [29]:
print(tokenized_dataset["input_ids"])

[[40, 423, 16555, 257, 6512, 561, 6129, 1497, 11, 198, 1870, 407, 1702, 416, 616, 2156, 477, 1110, 26, 198, 198, 11980, 537, 6320, 616, 2832, 379, 683, 422, 262, 3420, 198, 2215, 340, 3947, 355, 611, 314, 714, 6842, 645, 517, 13, 198, 198, 464, 8046, 1276, 11476, 423, 587, 287, 502, 13, 198, 464, 6512, 373, 407, 284, 8138, 329, 465, 1994, 13, 198, 198, 1870, 286, 1781, 612, 1276, 307, 1223, 2642, 198, 818, 10291, 284, 9550, 597, 3496, 13, 198, 198, 1544, 27771, 287, 262, 2344, 11, 290, 851, 644, 373, 326, 198, 21428, 287, 262, 17266, 2374, 11, 14005, 11, 475, 407, 257, 10905, 30, 198, 1544, 6204, 612, 6079, 2805, 1028, 465, 1807, 11, 198, 1870, 1865, 1165, 3492, 284, 1975, 262, 749, 13, 198, 198, 1, 5812, 11, 326, 338, 262, 20494, 12, 259, 12, 2436, 4207, 553, 314, 531, 26, 198, 1870, 4988, 340, 373, 3148, 1576, 329, 12734, 198, 18108, 356, 475, 287, 514, 284, 7048, 287, 9960, 198, 16678, 2330, 10632, 333, 3610, 286, 1737, 329, 16903, 13, 198, 198, 1135, 6204, 257, 2589, 523, 287, 257,

In [31]:
concatenated = sum(tokenized_dataset['input_ids'], [])
print(concatenated)

[40, 423, 16555, 257, 6512, 561, 6129, 1497, 11, 198, 1870, 407, 1702, 416, 616, 2156, 477, 1110, 26, 198, 198, 11980, 537, 6320, 616, 2832, 379, 683, 422, 262, 3420, 198, 2215, 340, 3947, 355, 611, 314, 714, 6842, 645, 517, 13, 198, 198, 464, 8046, 1276, 11476, 423, 587, 287, 502, 13, 198, 464, 6512, 373, 407, 284, 8138, 329, 465, 1994, 13, 198, 198, 1870, 286, 1781, 612, 1276, 307, 1223, 2642, 198, 818, 10291, 284, 9550, 597, 3496, 13, 198, 198, 1544, 27771, 287, 262, 2344, 11, 290, 851, 644, 373, 326, 198, 21428, 287, 262, 17266, 2374, 11, 14005, 11, 475, 407, 257, 10905, 30, 198, 1544, 6204, 612, 6079, 2805, 1028, 465, 1807, 11, 198, 1870, 1865, 1165, 3492, 284, 1975, 262, 749, 13, 198, 198, 1, 5812, 11, 326, 338, 262, 20494, 12, 259, 12, 2436, 4207, 553, 314, 531, 26, 198, 1870, 4988, 340, 373, 3148, 1576, 329, 12734, 198, 18108, 356, 475, 287, 514, 284, 7048, 287, 9960, 198, 16678, 2330, 10632, 333, 3610, 286, 1737, 329, 16903, 13, 198, 198, 1135, 6204, 257, 2589, 523, 287, 257, 

In [ ]:
tokenizer.encode("Hello <pad> world!", return_special_tokens_mask=True)

[15496, 1279, 15636, 29, 995, 0]

In [56]:
len(concatenated)

19896

In [ ]:
block_size = 128 

chunks = (len(concatenated) // block_size)
total=chunks*block_size
total

19840

In [58]:
block_size = 128  # You can also try 256 or 512 depending on memory
def group_texts(examples):
    # Concatenate all input_ids together into one long list
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // block_size) * block_size
    result = {
        "input_ids": [concatenated[i : i + block_size] for i in range(0, total_length, block_size)],
    }
    result["attention_mask"] = [[1] * block_size] * len(result["input_ids"])
    result["labels"] = result["input_ids"].copy()  # For language modeling, labels are the same as input_ids
    return result

# Apply grouping to the tokenized dataset
lm_datasets = tokenized_dataset.map(group_texts, batched=True)


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [59]:
lm_datasets

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 155
})

In [60]:
from transformers import GPT2LMHeadModel

# Load pre-trained GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")


In [61]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="poem_gpt2",            # where model checkpoints will be saved
    overwrite_output_dir=True,
    num_train_epochs=5,                  # change as needed
    per_device_train_batch_size=2,       # adjust based on your GPU/CPU
    save_steps=100,                      # how often to save model
    save_total_limit=2,                  # max checkpoints to keep
    logging_steps=10,
    prediction_loss_only=True
)


In [62]:
from transformers import Trainer, DataCollatorForLanguageModeling

# This collator handles padding and shifting labels for GPT-style training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # GPT-2 is a causal LM (not masked LM)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets,
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [63]:
trainer.train()


  0%|          | 0/390 [00:00<?, ?it/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 4.5627, 'grad_norm': 15.211960792541504, 'learning_rate': 4.871794871794872e-05, 'epoch': 0.13}
{'loss': 4.3218, 'grad_norm': 18.86245346069336, 'learning_rate': 4.7435897435897435e-05, 'epoch': 0.26}
{'loss': 4.3755, 'grad_norm': 15.773578643798828, 'learning_rate': 4.615384615384616e-05, 'epoch': 0.38}
{'loss': 4.2717, 'grad_norm': 15.526811599731445, 'learning_rate': 4.4871794871794874e-05, 'epoch': 0.51}
{'loss': 4.3225, 'grad_norm': 14.796587944030762, 'learning_rate': 4.358974358974359e-05, 'epoch': 0.64}
{'loss': 4.0834, 'grad_norm': 13.673797607421875, 'learning_rate': 4.230769230769231e-05, 'epoch': 0.77}
{'loss': 4.1906, 'grad_norm': 13.749070167541504, 'learning_rate': 4.1025641025641023e-05, 'epoch': 0.9}
{'loss': 4.2151, 'grad_norm': 13.239554405212402, 'learning_rate': 3.974358974358974e-05, 'epoch': 1.03}
{'loss': 3.8858, 'grad_norm': 13.897130012512207, 'learning_rate': 3.846153846153846e-05, 'epoch': 1.15}
{'loss': 3.8823, 'grad_norm': 14.377354621887207, 'lea

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 3.7386, 'grad_norm': 14.199477195739746, 'learning_rate': 2.948717948717949e-05, 'epoch': 2.05}
{'loss': 3.5821, 'grad_norm': 16.76866912841797, 'learning_rate': 2.8205128205128207e-05, 'epoch': 2.18}
{'loss': 3.5196, 'grad_norm': 14.188799858093262, 'learning_rate': 2.6923076923076923e-05, 'epoch': 2.31}
{'loss': 3.3903, 'grad_norm': 18.924076080322266, 'learning_rate': 2.564102564102564e-05, 'epoch': 2.44}
{'loss': 3.4886, 'grad_norm': 15.884685516357422, 'learning_rate': 2.435897435897436e-05, 'epoch': 2.56}
{'loss': 3.5926, 'grad_norm': 15.884355545043945, 'learning_rate': 2.307692307692308e-05, 'epoch': 2.69}
{'loss': 3.5901, 'grad_norm': 17.679378509521484, 'learning_rate': 2.1794871794871795e-05, 'epoch': 2.82}
{'loss': 3.7275, 'grad_norm': 17.259172439575195, 'learning_rate': 2.0512820512820512e-05, 'epoch': 2.95}


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 3.4884, 'grad_norm': 17.37778663635254, 'learning_rate': 1.923076923076923e-05, 'epoch': 3.08}
{'loss': 3.3877, 'grad_norm': 15.273229598999023, 'learning_rate': 1.794871794871795e-05, 'epoch': 3.21}
{'loss': 3.344, 'grad_norm': 16.65895652770996, 'learning_rate': 1.6666666666666667e-05, 'epoch': 3.33}
{'loss': 3.3806, 'grad_norm': 17.03577995300293, 'learning_rate': 1.5384615384615387e-05, 'epoch': 3.46}
{'loss': 3.411, 'grad_norm': 18.57986831665039, 'learning_rate': 1.4102564102564104e-05, 'epoch': 3.59}
{'loss': 3.4685, 'grad_norm': 17.037105560302734, 'learning_rate': 1.282051282051282e-05, 'epoch': 3.72}
{'loss': 3.2781, 'grad_norm': 16.810930252075195, 'learning_rate': 1.153846153846154e-05, 'epoch': 3.85}
{'loss': 3.3498, 'grad_norm': 16.584285736083984, 'learning_rate': 1.0256410256410256e-05, 'epoch': 3.97}


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 3.4133, 'grad_norm': 15.431316375732422, 'learning_rate': 8.974358974358976e-06, 'epoch': 4.1}
{'loss': 3.2839, 'grad_norm': 19.020294189453125, 'learning_rate': 7.692307692307694e-06, 'epoch': 4.23}
{'loss': 3.2023, 'grad_norm': 15.703286170959473, 'learning_rate': 6.41025641025641e-06, 'epoch': 4.36}
{'loss': 3.3347, 'grad_norm': 17.0230770111084, 'learning_rate': 5.128205128205128e-06, 'epoch': 4.49}
{'loss': 3.3161, 'grad_norm': 17.997146606445312, 'learning_rate': 3.846153846153847e-06, 'epoch': 4.62}
{'loss': 3.1516, 'grad_norm': 16.855833053588867, 'learning_rate': 2.564102564102564e-06, 'epoch': 4.74}
{'loss': 3.3105, 'grad_norm': 17.098655700683594, 'learning_rate': 1.282051282051282e-06, 'epoch': 4.87}
{'loss': 3.3239, 'grad_norm': 23.30533790588379, 'learning_rate': 0.0, 'epoch': 5.0}
{'train_runtime': 727.3066, 'train_samples_per_second': 1.066, 'train_steps_per_second': 0.536, 'train_loss': 3.6714717962802985, 'epoch': 5.0}


TrainOutput(global_step=390, training_loss=3.6714717962802985, metrics={'train_runtime': 727.3066, 'train_samples_per_second': 1.066, 'train_steps_per_second': 0.536, 'total_flos': 50625331200000.0, 'train_loss': 3.6714717962802985, 'epoch': 5.0})

In [65]:
trainer.save_model("poem")
tokenizer.save_pretrained("poem")


('poem\\tokenizer_config.json',
 'poem\\special_tokens_map.json',
 'poem\\vocab.json',
 'poem\\merges.txt',
 'poem\\added_tokens.json',
 'poem\\tokenizer.json')

In [66]:
from transformers import pipeline

generator = pipeline("text-generation", model="poem", tokenizer=tokenizer)

# Try generating!
lines= generator("love", max_length=100, num_return_sequences=1)


In [67]:
print(lines[0]['generated_text'])

love by the shore,
He thinks, I dare say, so far east
That I scarcely am free to catch them,
And, perhaps, in this winter?
A solitary wild-camouflage lie,
Which the moon has at my feet,
Yet not of the world to behold!

And on this shore he lies all day lonely!
He lives in an inner cottage with his father,
With his mother being alone;
He is his father's only
